# MCTS library survey

- Four libraries are actually run here, on the same `TicTacToe` from
  `alphazero_utils.py`: `monte-carlo-tree-search`, `mctspy`, `mcts-simple`,
  `imparaai-montecarlo`
- Two libraries (`mctx`, `mctorch-mcts`) are built for batched, at-scale
  search rather than small demos; their own usage code is shown but not
  executed here
- All adapter code lives in `mcts_library_survey_utils.py`; this notebook
  only calls into it

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import logging

import helpers.hdbg as hdbg
import helpers.hnotebook as hnotebook

_LOG = logging.getLogger(__name__)

hdbg.init_logger(verbosity=logging.INFO)
hnotebook.config_notebook()

In [ ]:
import research.Implement_MonteCarlo_Tree_Search_and_Alpha_Zero.alphazero_utils as rimtsaazau
import research.Implement_MonteCarlo_Tree_Search_and_Alpha_Zero.mcts_library_survey_utils as rimtsaazmlsu

# Part 1: The six libraries

They split into three tiers rather than six equivalent options:

| Tier | Library | Character |
|---|---|---|
| Educational, small trees | `monte-carlo-tree-search` | Maintained PyPI continuation of int8's minimal reference MCTS; intended for small game trees |
| Educational, small trees | `mctspy` | Same tier, ships tic-tac-toe/Connect4 examples out of the box |
| Educational -> RL | `mcts-simple` | Rollout MCTS + variants, RL-oriented, seeded RNG, msgpack persistence |
| Policy-aware | `imparaai-montecarlo` | You plug in `child_finder`/`node_evaluator` hooks, so it natively supports neural-network-guided ("expert policy") search, not just rollouts |
| Production, batched | `mctx` (DeepMind) | JAX-native, JIT-compiled, searches batches of positions in parallel; what real AlphaZero/MuZero training uses |
| Production, batched, new | `mctorch-mcts` | PyTorch + C++/Cython accelerated backend, released July 2026; ships a JAX/`mctx` reference backend of its own for benchmarking |

The first four are plain pip installs and are actually run below. The last
two are built for batched, GPU/TPU-scale search rather than a quick demo, and
`mctorch-mcts` additionally needs a C++ toolchain and CMake to build, so
adding them here would meaningfully grow this project's Docker image for a
one-off demo. Their own usage code is shown in Part 4 without running it.

# Part 2: Four libraries

## Cell 2.1: The same forced-win position, four engines

Same position we used to sanity-check our own MCTS in `main.ipynb`: X has two
in a row, so the correct move is closing cell 2. Each library gets its own
adapter in `mcts_library_survey_utils.py` translating our `TicTacToe` into
whatever state/game protocol that library expects.

In [ ]:
game = rimtsaazau.TicTacToe()
demo_state = (1, 1, 0, -1, -1, 0, 0, 0, 0)
print(game.render(demo_state))
print()

for library_name, player in rimtsaazmlsu.LIBRARY_PLAYERS.items():
    move = player(game, demo_state)
    print(f"{library_name}: move {move}")

**Key observations**:
- All four libraries take the same `game`/`state` from `alphazero_utils.py`;
  only the adapter around each call changes
- A single easy position isn't enough to tell libraries apart reliably (with
  200 simulations, an obviously winning move is usually found by all of
  them); Part 3 runs many full games instead

# Part 3: Head-to-head vs. a random player

## Cell 3.1: Win rate over 30 games each

Same evaluation as `main.ipynb`: each library plays 30 full games as player
`1` against `alphazero_utils.random_player`, all at the same 200-simulation
budget, alongside our own MCTS for reference.

In [ ]:
comparison_df = rimtsaazmlsu.evaluate_all_libraries(num_games=30)
display(comparison_df)

**Key observations**:
- `monte-carlo-tree-search`, `mctspy`, and `mcts-simple` land in the same
  range as our own MCTS: near-100% win rate, essentially no losses
- `imparaai-montecarlo` consistently trails the other three in repeated runs
  of this evaluation (roughly 60-70% win rate here, vs. 0% losses for
  everyone else). The likely cause: once its search reaches an
  already-resolved (terminal) branch early, re-expanding that branch is a
  no-op in the library's `expand()` method, so its visit count stops
  growing; with a modest simulation budget this can leave several root
  moves tied on visit count, and the final `make_choice()` breaks ties at
  random instead of favoring the resolved winning branch

# Part 4: Two libraries built for scale (not run here)

Code below is copied from each project's own documentation, not executed in
this notebook.

## `mctx` (google-deepmind)

Source: [github.com/google-deepmind/mctx](https://github.com/google-deepmind/mctx)

```python
policy_output = mctx.gumbel_muzero_policy(params, rng_key, root, recurrent_fn,
                                          num_simulations=32)
```

## `mctorch-mcts`

Source: [pypi.org/project/mctorch-mcts](https://pypi.org/project/mctorch-mcts/)
(package README, `mctorch-mcts` 0.1.0)

```python
import torch
import torch.nn as nn
from mctorch import alphazero_policy, RootFnOutput, RecurrentFnOutput

# Minimal two-headed network (policy + value)
class Net(nn.Module):
    def forward(self, obs):
        ...  # return policy_logits [B, A], value [B]

net = Net()
board = torch.zeros(16, 9)          # 16 games, 9-cell TicTacToe boards

with torch.no_grad():
    logits, values = net(board)

root = RootFnOutput(prior_logits=logits, value=values, embedding=board)

def recurrent_fn(params, actions, embedding):
    new_board, reward, discount = your_step_fn(embedding, actions)
    with torch.no_grad():
        new_logits, new_values = params(new_board)
    return RecurrentFnOutput(reward=reward, discount=discount,
                             prior_logits=new_logits, value=new_values), new_board

out = alphazero_policy(net, root, recurrent_fn, num_simulations=200)
print(out.action)        # [B] best action per game
```

# Part 5: Comparison and recommendation

| Type | Name | Description | Website | Strength |
|---|---|---|---|---|
| Educational | `monte-carlo-tree-search` | Minimal `BaseState`/`MCTS` interface, matched our own win rate | https://pypi.org/project/monte-carlo-tree-search/ | Smallest interface to learn from |
| Educational | `mctspy` | Minimal, ships tic-tac-toe/Connect4 examples, matched our own win rate | https://pypi.org/project/mctspy/ | Fastest to get a working example from |
| Educational -> RL | `mcts-simple` | Rollout MCTS with an explicit `self_play()` training loop, matched our own win rate | https://github.com/DenseLance/mcts-simple | Persistence (save/load a trained tree) |
| Policy-aware | `imparaai-montecarlo` | `child_finder`/`node_evaluator` hooks support neural-policy search; underperformed on plain rollouts in our test | https://github.com/ImparaAI/monte-carlo-tree-search | Only one here designed to plug in a neural net without forking the library |
| Production, batched | `mctx` | JAX-native, JIT-compiled, batched search; powers real AlphaZero/MuZero/Gumbel-MuZero training | https://github.com/google-deepmind/mctx | Mature, maintained by DeepMind, built for scale |
| Production, batched, new | `mctorch-mcts` | PyTorch + C++/Cython backend, month-old, ships its own `mctx` reference backend | https://pypi.org/project/mctorch-mcts/ | PyTorch-native alternative to `mctx`, unproven |

**When to use which**:

* Use `monte-carlo-tree-search` or `mctspy` if you want a small implementation that is easy to read and learn from.
* Use `mcts-simple` if you need to save and reload the tree/model or want something that fits naturally into an RL training loop.
* Use `imparaai-montecarlo` if you want to plug in a neural network's policy and value outputs without implementing the tree search yourself. Based on Part 3, expect to need more simulations than with a plain rollout baseline.
* Use `mctx` if you need high throughput, such as batched self-play over many positions on a GPU or TPU. It is designed for AlphaZero-style training rather than small educational examples.
* Use `mctorch-mcts` if you want a PyTorch-based alternative to `mctx`. It is still new, requires a CMake build from source, and currently only provides a working prebuilt wheel for Windows.

**Tying back to the project roadmap**: the educational tier matches
Milestone 1 (pure MCTS, this project's `alphazero_utils.py`);
`imparaai-montecarlo`'s policy hook and `mctx`/`mctorch-mcts`'s batched
search both point at what Milestone 2 (AlphaZero/MuZero) will actually need.